# Laser Execute

This notebook uses the Poetry-managed environment. All dependencies from `pyproject.toml` are available.


In [1]:
# Verify we're using the correct environment
import sys
print(f"Python: {sys.executable}")


Python: /Users/smagny/Femtum/Femtum-API-Examples/python/.venv/bin/python


In [2]:
from femtum_sdk.api.contracts.v1.hardware.health_pb2 import HealthRequest
from femtum_sdk.api.trimming import TrimmingApi

api = TrimmingApi(host="localhost:5210")
api.__enter__()
health_response = api.health.GetHealth(HealthRequest())
print(health_response)


AttributeError: 'TrimmingApi' object has no attribute 'health'

## Execution Simulation

In [ ]:
from femtum_sdk.api.contracts.v1.dto.measure_pb2 import DistanceMeasure, DistanceMeasureUnit, EnergyMeasure, EnergyMeasureUnit
from femtum_sdk.api.contracts.v1.trimming.laser_execution_pb2 import LaserExecutionDistance, SimulateDatabaseRecipeRequest, SimulateDatabaseRecipeResponse, LaserExecutionOptions


simulation_iteration_1_response: SimulateDatabaseRecipeResponse = api.laser_execution.SimulateDatabaseRecipe(SimulateDatabaseRecipeRequest(
    RecipeId="4",
    Options=LaserExecutionOptions(
        CalibrationId="3",
        Distance=LaserExecutionDistance(
            RelativeFromPreviousExecution=DistanceMeasure(
                Value=0.0,
                Unit=DistanceMeasureUnit.Microns,
            ),
        ),
        EnergyPerPulse=EnergyMeasure(
            Value=0.2,
            Unit=EnergyMeasureUnit.Nanojoules,
        ),
    ),
    PreviousExecutions=[],
))

simulation_iteration_2_response: SimulateDatabaseRecipeResponse = api.laser_execution.SimulateDatabaseRecipe(SimulateDatabaseRecipeRequest(
    RecipeId="4",
    Options=LaserExecutionOptions(
        CalibrationId="3",
        Distance=LaserExecutionDistance(
            RelativeFromPreviousExecution=DistanceMeasure(
                Value=10.0,
                Unit=DistanceMeasureUnit.Microns,
            ),
        ),
        EnergyPerPulse=EnergyMeasure(
            Value=0.2,
            Unit=EnergyMeasureUnit.Nanojoules,
        ),
    ),
    PreviousExecutions=simulation_iteration_1_response.Executions,
))

print(simulation_iteration_2_response.Executions)



## Execute With Laser

In [ ]:
from femtum_sdk.api.contracts.v1.trimming.laser_execution_pb2 import ExecuteDatabaseRecipeRequest, ExecuteDatabaseRecipeResponse


response: ExecuteDatabaseRecipeResponse = api.laser_execution.ExecuteDatabaseRecipe(ExecuteDatabaseRecipeRequest(
    RecipeId="4",
    ComponentPath="MyComponent",
    Options=LaserExecutionOptions(
        LaserEnergyCalibrationId="3",
        Distance=LaserExecutionDistance(
            RelativeFromPreviousExecution=DistanceMeasure(
                Value=0.0,
                Unit=DistanceMeasureUnit.Microns,
            ),
        ),
        EnergyPerPulse=EnergyMeasure(
            Value=0.2,
            Unit=EnergyMeasureUnit.Nanojoules,
        ),
    ),
))
execution = response.Execution

## Fetch executions

In [ ]:
from femtum_sdk.api import DatastoreApi
from femtum_sdk.api.contracts.v1.storage.laser_execution_pb2 import LaserExecutionPage, LaserExecutionsFilterRequest, ListByPageLaserExecutionsRequest
from femtum_sdk.api.contracts.v1.storage.query_pb2 import QueryField, QueryOperator


datastore_api = DatastoreApi(hostUrl="localhost:5208")
datastore_api.__enter__()
is_up = datastore_api.health.is_up()
print(f"Is Datastore API Up: {is_up}")
assert is_up

page: LaserExecutionPage = datastore_api.laser_execution.ListByPage(ListByPageLaserExecutionsRequest(Filters=LaserExecutionsFilterRequest(ComponentPath=QueryField(Value="MyComponent", Operator=QueryOperator.EQUALS))))
assert any(item.Id == execution.Id for item in page.Items)

In [ ]:
api.__exit__()
